[Python, Visually](https://johnfisher-ai.github.io/Python-Visual-Guides/) &nbsp;&rsaquo;&nbsp; [Files, Paths and Formats](https://johnfisher-ai.github.io/Python-Visual-Guides/files-paths-and-formats.html)

# Encodings &middot; Solutions


One way to do each task. Not the only way. If yours runs and does what was asked, yours is
right too.


In [1]:
from pathlib import Path
import shutil

scratch = Path("scratch")
scratch.mkdir(exist_ok=True)


**1.** Print the number of characters in `"na\u00efve"` and the number of bytes it becomes
in `utf-8` and in `latin-1`.


In [2]:
word = "na\u00efve"

print("characters:", len(word))
print("utf-8:     ", len(word.encode("utf-8")), "bytes", word.encode("utf-8"))
print("latin-1:   ", len(word.encode("latin-1")), "bytes", word.encode("latin-1"))


characters: 5
utf-8:      6 bytes b'na\xc3\xafve'
latin-1:    5 bytes b'na\xefve'


Five characters, six bytes in UTF-8, five in Latin-1. The accented character is the one that
costs two bytes, and every other character costs one in both.


**2.** Encode `"se\u00f1or"` as UTF-8 and decode the result as Latin-1.


In [3]:
correct = "se\u00f1or"
mojibake = correct.encode("utf-8").decode("latin-1")

print("correct: ", correct)
print("mojibake:", mojibake)
print("lengths: ", len(correct), "against", len(mojibake))


correct:  señor
mojibake: seÃ±or
lengths:  5 against 6


`señor` became `seÃ±or`, and it gained a character. The two bytes UTF-8 used for `ñ` were read
as two separate Latin-1 characters.


**3.** Repair the string from task 2.


In [4]:
repaired = mojibake.encode("latin-1").decode("utf-8")

print(repaired)
print("matches the original:", repaired == correct)


señor
matches the original: True


Encode with the rule that was wrongly used for decoding, which gets the original bytes back,
then decode with the right one. Nothing was lost on the way in, which is why this works.


**4.** Write `"Z\u00fcrich\n"` as Latin-1, then try to read it as UTF-8 and print the
exception instead of letting it stop the cell.


In [5]:
p = scratch / "zurich.txt"
p.write_bytes("Z\u00fcrich\n".encode("latin-1"))

try:
    p.read_text(encoding="utf-8")
except UnicodeDecodeError as e:
    print(type(e).__name__)
    print(e)


UnicodeDecodeError
'utf-8' codec can't decode byte 0xfc in position 1: invalid start byte


The message names the byte, `0xfc`, and its position, 1. In a file of a few lines that is enough
to find it by eye; in a large one it tells you whether the whole file is the wrong encoding or
only one row is unusual.


**5.** Read the same file three ways and print all three with `repr`.


In [6]:
print("ignore: ", repr(p.read_text(encoding="utf-8", errors="ignore")))
print("replace:", repr(p.read_text(encoding="utf-8", errors="replace")))
print("correct:", repr(p.read_text(encoding="latin-1")))


ignore:  'Zrich\n'
replace: 'Z�rich\n'
correct: 'Zürich\n'


`ignore` produced `'Zrich\n'`, which is a perfectly ordinary looking string and is a city that
does not exist. Nothing downstream could detect the missing character.

`replace` produced a visible marker. The correct encoding produced the actual name.


**6.** Write `"id,name\n1,ada\n"` with `encoding="utf-8-sig"`, read it back both ways, and
print whether the first column equals `"id"`.


In [7]:
csv_file = scratch / "export.csv"
csv_file.write_text("id,name\n1,ada\n", encoding="utf-8-sig")

as_utf8 = csv_file.read_text(encoding="utf-8")
as_sig = csv_file.read_text(encoding="utf-8-sig")

print("utf-8     first column:", repr(as_utf8.split(",")[0]), "==", as_utf8.split(",")[0] == "id")
print("utf-8-sig first column:", repr(as_sig.split(",")[0]), "==", as_sig.split(",")[0] == "id")


utf-8     first column: '\ufeffid' == False
utf-8-sig first column: 'id' == True


`'\ufeffid'` against `'id'`. The byte order mark became part of the first column name, and it is
invisible everywhere except in `repr`.

This is the whole explanation for a `KeyError` on a column that is plainly there. Reading Excel
exports with `encoding="utf-8-sig"` avoids it, and costs nothing when the mark is absent.


In [8]:
shutil.rmtree(scratch)

print("cleaned up:", not scratch.exists())


cleaned up: True


---

&#8592; **Back to:** [Encodings](https://colab.research.google.com/github/johnfisher-ai/Python-Visual-Guides/blob/main/notebooks/files-paths-and-formats/03-encodings.ipynb)  &nbsp;&middot;&nbsp;  [Files, Paths and Formats Notebooks](https://johnfisher-ai.github.io/Python-Visual-Guides/files-paths-and-formats.html)
